In [4]:
# Q1

import time
import torchvision
from torchvision import transforms
from d2l import torch as d2l

class FashionMNIST(d2l.DataModule):
    def __init__(self, batch_size=64, resize=(28, 28)):
        super().__init__()
        self.save_hyperparameters()
        trans = transforms.Compose([transforms.Resize(resize),
                                    transforms.ToTensor()])
        self.train = torchvision.datasets.FashionMNIST(
            root=self.root, train=True, transform=trans, download=True)
        self.val = torchvision.datasets.FashionMNIST(
            root=self.root, train=False, transform=trans, download=True)

    def get_dataloader(self, train):
        data = self.train if train else self.val
        return __import__('torch').utils.data.DataLoader(
            data, self.batch_size, shuffle=train,
            num_workers=self.num_workers)

In [5]:
batch_sizes = [1, 8, 64, 256]
results = {}

for bs in batch_sizes:
    d = FashionMNIST(batch_size=bs, resize=(32, 32))
    tic = time.time()
    for X, y in d.train_dataloader():
        continue
    elapsed = time.time() - tic
    results[bs] = elapsed
    print(f'batch_size={bs:>4d}  batches={len(d.train)//bs:>5d}  time={elapsed:.2f}s')

batch_size=   1  batches=60000  time=27.03s
batch_size=   8  batches= 7500  time=10.18s
batch_size=  64  batches=  937  time=8.68s
batch_size= 256  batches=  234  time=8.71s


In [6]:
# Q2
import torch, cProfile, pstats, io

d_base = FashionMNIST(batch_size=64, resize=(32, 32))
pr = cProfile.Profile()
pr.enable()
for X, y in d_base.train_dataloader(): continue
pr.disable()

buf = io.StringIO()
pstats.Stats(pr, stream=buf).sort_stats('cumulative').print_stats(8)
for line in buf.getvalue().splitlines():
    print(line[:80])

configs = {
    'baseline (num_workers=0)':  dict(num_workers=0, pin_memory=False, persistent_workers=False),
    'num_workers=4':             dict(num_workers=4, pin_memory=False, persistent_workers=False),
    '+ pin_memory':              dict(num_workers=4, pin_memory=True,  persistent_workers=False),
    '+ persistent_workers':      dict(num_workers=4, pin_memory=True,  persistent_workers=True),
}
print("\n--- optimization comparison ---")
for name, kw in configs.items():
    loader = torch.utils.data.DataLoader(d_base.train, batch_size=64, shuffle=True, **kw)
    tic = time.time()
    for X, y in loader: continue
    print(f'{name:<30s} {time.time()-tic:.2f}s')

         246388 function calls (241687 primitive calls) in 9.144 seconds

   Ordered by: cumulative time
   List reduced from 341 to 8 due to restriction <8>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000    9.195    4.598 c:\Users\24789\miniconda3\envs\pyt
        2    0.000    0.000    9.195    4.598 {built-in method builtins.exec}
      939    0.009    0.000    9.087    0.010 c:\Users\24789\miniconda3\envs\pyt
      939    0.009    0.000    8.974    0.010 c:\Users\24789\miniconda3\envs\pyt
      938    0.001    0.000    8.018    0.009 c:\Users\24789\miniconda3\envs\pyt
      939    0.002    0.000    8.017    0.009 c:\Users\24789\miniconda3\envs\pyt
      939    0.007    0.000    8.015    0.009 c:\Users\24789\miniconda3\envs\pyt
      939    0.002    0.000    7.802    0.008 c:\Users\24789\miniconda3\envs\pyt



--- optimization comparison ---
baseline (num_workers=0)       5.03s
num_workers=4                  9.22s
+ pin_memory   